In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [13]:
data = pd.read_csv('../datasets/Iris.csv')
species_map = {'Iris-setosa': 0, 'Iris-versicolor': 1, 'Iris-virginica': 2}
data['Species'] = data['Species'].map(species_map)
data.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,0
1,2,4.9,3.0,1.4,0.2,0
2,3,4.7,3.2,1.3,0.2,0
3,4,4.6,3.1,1.5,0.2,0
4,5,5.0,3.6,1.4,0.2,0


In [14]:
X_data = torch.tensor(data.drop(columns=['Id', 'Species']).values, dtype=torch.float32)
y_data = torch.tensor(data['Species'].values, dtype=torch.long)   # long, not float

X_train, X_test, y_train, y_test = train_test_split(
  X_data, y_data, test_size=0.2, random_state=42, stratify=y_data
)

In [16]:
y_train

tensor([0, 2, 1, 0, 1, 2, 1, 2, 2, 2, 2, 1, 1, 1, 1, 0, 0, 2, 2, 0, 1, 0, 2, 0,
        1, 2, 2, 0, 2, 0, 0, 1, 1, 0, 2, 2, 1, 1, 2, 1, 0, 1, 0, 2, 0, 0, 2, 0,
        0, 0, 0, 1, 2, 1, 0, 2, 1, 2, 0, 2, 0, 1, 2, 0, 1, 1, 2, 1, 1, 2, 0, 0,
        0, 2, 1, 2, 1, 2, 2, 1, 0, 2, 1, 0, 2, 0, 2, 1, 1, 0, 1, 2, 0, 0, 2, 2,
        2, 1, 2, 0, 2, 1, 2, 2, 0, 1, 1, 1, 1, 1, 0, 2, 1, 1, 0, 0, 0, 0, 1, 0])

In [28]:
class IrisDetectionModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.network = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 3),
        )
    def forward(self, x):
        return self.network(x)

In [33]:
# --------------------------------------------------
# 4. Training
# --------------------------------------------------

torch.manual_seed(42)

model = IrisDetectionModel()


for name, param in model.named_parameters():
    print(name)
    print(param)
    print("Shape:", param.shape)
    print()


total_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total trainable parameters:", total_params)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


EPOCHS = 1000
losses = []

for epoch in range(EPOCHS):
    # Clear old gradients
    optimizer.zero_grad()
    
    # Forward pass
    output = model(X_train)
    
    # calculate loss
    loss = criterion(output, y_train)
    
    #Back propagation
    loss.backward()
    
    # Update weights
    optimizer.step()

    losses.append(loss.item()) 
    
    if epoch % 50 == 0:
        print(f"Epoch {epoch:4d} | Loss: {loss.item():.4f}")
    

network.0.weight
Parameter containing:
tensor([[ 0.3823,  0.4150, -0.1171,  0.4593],
        [-0.1096,  0.1009, -0.2434,  0.2936],
        [ 0.4408, -0.3668,  0.4346,  0.0936],
        [ 0.3694,  0.0677,  0.2411, -0.0706],
        [ 0.3854,  0.0739, -0.2334,  0.1274],
        [-0.2304, -0.0586, -0.2031,  0.3317],
        [-0.3947, -0.2305, -0.1412, -0.3006],
        [ 0.0472, -0.4938,  0.4516, -0.4247],
        [ 0.3860,  0.0832, -0.1624,  0.3090],
        [ 0.0779,  0.4040,  0.0547, -0.1577],
        [ 0.1343, -0.1356,  0.2104,  0.4464],
        [ 0.2890, -0.2186,  0.2886,  0.0895],
        [ 0.2539, -0.3048, -0.4950, -0.1932],
        [-0.3835,  0.4103,  0.1440,  0.2071],
        [ 0.1581, -0.0087,  0.3913, -0.3553],
        [ 0.0315, -0.3413,  0.1542, -0.1722]], requires_grad=True)
Shape: torch.Size([16, 4])

network.0.bias
Parameter containing:
tensor([ 0.1532, -0.1042,  0.4147, -0.2964, -0.2982, -0.2982,  0.4497,  0.1666,
         0.4811, -0.4126, -0.4959, -0.3912, -0.3363,  0.202